# GRF Training Notebook
**Google Research Football** — `academy_3_vs_1_with_keeper`

Installs gfootball from source (Linux process from `compile_engine.md`),
runs MAPPO training, and saves results to Google Drive for `mappo_demo.ipynb`.

## Workflow
1. Edit config (Cell 1).
2. Run all cells top-to-bottom.
3. Results are saved to Google Drive automatically.
4. `mappo_demo.ipynb` loads them via `DEMO_MODE=True`.

In [ ]:
# ================================================================
# CONFIGURATION — edit before running
# ================================================================
GITHUB_URL    = 'https://github.com/keckster00/mappo'
DRIVE_RESULTS = '/content/drive/MyDrive/mappo_demo'  # must match mappo_demo.ipynb
GRF_VENV      = '/content/football-env'               # Python venv for gfootball
GRF_SRC       = '/content/football'                   # gfootball source clone

GRF_STEPS = 5_000_000   # ~30-60 min on A100
N_THREADS = 64           # parallel envs

print(f'GRF_STEPS : {GRF_STEPS:,}')
print(f'N_THREADS : {N_THREADS}')

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Drive mounted. Results folder: {DRIVE_RESULTS}')

REPO_PATH = '/content/mappo'
if not os.path.exists(REPO_PATH):
    ret = os.system(f'git clone {GITHUB_URL} {REPO_PATH}')
    if ret != 0:
        raise RuntimeError('git clone failed — check GITHUB_URL in the config cell.')
else:
    os.system(f'git -C {REPO_PATH} pull origin main')
print(f'Repo: {REPO_PATH}')

## GRF Setup
Follows the **Linux** process from `compile_engine.md`:
1. Install system packages via `apt-get`
2. Clone GRF source from `github.com/google-research/football`
3. Create a Python 3 venv and upgrade pip/setuptools/wheel
4. `pip install psutil` inside the venv
5. `pip install .` from the cloned directory (cmake runs automatically via setup.py)
6. Install training dependencies (torch, gym, etc.)

**Run once per Colab session** (~8–12 min total).

In [ ]:
import subprocess, os

VENV_PY  = f'{GRF_VENV}/bin/python'
VENV_PIP = f'{GRF_VENV}/bin/pip'

def sh(cmd, label='', cwd=None, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd, env=env)
    tag = label or cmd[:70]
    if r.returncode == 0:
        print(f'  [OK]   {tag}')
    else:
        out = ((r.stdout or '') + (r.stderr or '')).strip()
        print(f'  [FAIL] {tag}')
        if out:
            print(out[-3000:])
        raise RuntimeError(f'Failed: {tag}')

# ── 1. System packages ────────────────────────────────────────────────────────────
# python3.10 + python3.10-venv required: Colab runs 3.12 but libboost-all-dev
# only ships libboost_python310 on Ubuntu 22.04. The venv must use 3.10 to match.
print('Step 1: System packages...')
sh('apt-get update -qq', 'apt-get update')
sh(
    'DEBIAN_FRONTEND=noninteractive apt-get install -y -q '
    'git cmake build-essential libgl1-mesa-dev libsdl2-dev '
    'libsdl2-image-dev libsdl2-ttf-dev libsdl2-gfx-dev libboost-all-dev '
    'libdirectfb-dev libst-dev mesa-utils xvfb x11vnc python3-pip '
    'python3.10 python3.10-venv python3.10-dev',
    'system packages'
)

# ── 2. Clone GRF source ─────────────────────────────────────────────────────────
print('\nStep 2: Clone GRF source...')
if not os.path.exists(f'{GRF_SRC}/setup.py'):
    sh(
        f'git clone --recurse-submodules '
        f'https://github.com/google-research/football {GRF_SRC}',
        'git clone GRF'
    )
else:
    print('  [OK]   already cloned')

# ── 3. Create Python 3.10 venv ───────────────────────────────────────────────────
# Use python3.10 explicitly so cmake detects Python 3.10 and links against
# libboost_python310 (the only boost-python available on Ubuntu 22.04).
print('\nStep 3: Create Python 3.10 venv...')
if not os.path.exists(VENV_PY):
    sh(f'python3.10 -m venv {GRF_VENV}', 'create venv (python3.10)')
else:
    print('  [OK]   venv exists')
sh(f'{VENV_PIP} install -q --upgrade pip setuptools wheel', 'pip upgrade')
sh(f'{VENV_PIP} install -q psutil', 'psutil')

# ── 4. Build gfootball (∼5-10 min, cmake output streamed) ─────────────────────
print('\nStep 4: Build gfootball (∼5-10 min, cmake output below)...')
result = subprocess.run(f'{VENV_PIP} install .', shell=True, cwd=GRF_SRC)
if result.returncode != 0:
    raise RuntimeError('gfootball build failed — see output above.')
print('  [OK]   gfootball installed')

# ── 5. Training dependencies ────────────────────────────────────────────────
print('\nStep 5: Training dependencies...')
sh(f'{VENV_PIP} install -q gym==0.25.2', 'gym 0.25.2')
sh(
    f'{VENV_PIP} install -q torch torchvision '
    f'--index-url https://download.pytorch.org/whl/cu121',
    'torch + cuda'
)
sh(
    f'{VENV_PIP} install -q '
    f'numpy==1.26.4 scipy imageio tensorboard tensorboardX setproctitle wandb',
    'misc deps'
)
sh(f'{VENV_PIP} install -q -e {REPO_PATH}', 'onpolicy (editable)')

print('\nAll setup steps complete.')

In [ ]:
import subprocess

VENV_PY = f'{GRF_VENV}/bin/python'

def check(label, code):
    r = subprocess.run([VENV_PY, '-c', code], capture_output=True, text=True)
    ok = r.returncode == 0
    out = (r.stdout + r.stderr).strip().splitlines()
    status = 'OK' if ok else 'FAIL'
    first = out[0] if out else ''
    print(f'  [{status}] {label}: {first}')
    if not ok:
        full = (r.stdout + r.stderr).strip()
        if full:
            print(full[-1000:])
    return ok

all_ok = True
all_ok &= check('gfootball (import)', 'import gfootball; print(gfootball.__version__)')
all_ok &= check('gfootball (env)', "import gfootball.env as fe; e=fe.create_environment(env_name='academy_empty_goal',representation='simple115v2'); e.close(); print('env OK')")
all_ok &= check('torch', 'import torch; print(torch.__version__, "| cuda:", torch.cuda.is_available())')
all_ok &= check('gym', 'import gym; print(gym.__version__)')
all_ok &= check('onpolicy', "import onpolicy; print('OK')")

if not all_ok:
    raise RuntimeError('Checks failed — fix errors above before training.')
print('\nAll checks passed. Ready to train.')

## Training
A smoke test (1 thread, 2 000 steps, ~30 s) runs first to catch errors
before committing to the full run.

Full run defaults: **5 M steps × 64 threads** — ~30–60 min on A100.
Adjust `GRF_STEPS` and `N_THREADS` in Cell 1.

In [ ]:
import subprocess, os, tempfile

os.environ['WANDB_MODE']     = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'

VENV_PY    = f'{GRF_VENV}/bin/python'
GRF_SCRIPT = f'{REPO_PATH}/onpolicy/scripts/train/train_football.py'

# GRF requires a virtual display even for headless training
os.system('pkill Xvfb 2>/dev/null; Xvfb :99 -screen 0 1280x720x24 &')

env = os.environ.copy()
env['DISPLAY']    = ':99'
env['PYTHONPATH'] = REPO_PATH + ':' + env.get('PYTHONPATH', '')

COMMON_ARGS = [
    '--env_name',           'Football',
    '--scenario_name',      'academy_3_vs_1_with_keeper',
    '--num_agents',         '3',
    '--algorithm_name',     'rmappo',
    '--seed',               '1',
    '--n_training_threads', '1',
    '--num_mini_batch',     '1',
    '--episode_length',     '200',
    '--ppo_epoch',          '15',
    '--use_ReLU',
    '--gain',               '0.01',
    '--lr',                 '5e-4',
    '--critic_lr',          '5e-4',
    '--representation',     'simple115v2',
    '--rewards',            'scoring',
    '--use_wandb',  # store_false action: passing this flag disables wandb, uses TensorBoard
]

def run_train(extra_args, label):
    cmd = [VENV_PY, GRF_SCRIPT] + COMMON_ARGS + extra_args
    with tempfile.NamedTemporaryFile(mode='w', suffix='.log', delete=False) as f:
        log_path = f.name
    with open(log_path, 'w') as f:
        result = subprocess.run(cmd, stderr=f, text=True, env=env)
    ok = result.returncode == 0
    if ok:
        print(f'{label}: PASSED')
        os.unlink(log_path)
    else:
        print(f'{label}: FAILED (exit {result.returncode})')
        with open(log_path) as f:
            lines = f.readlines()
        print('--- last 60 lines of stderr ---')
        print(''.join(lines[-60:]))
        os.unlink(log_path)
    return ok

print('Running smoke test (1 thread, 2 000 steps)...')
ok = run_train([
    '--experiment_name',   'smoke',
    '--n_rollout_threads', '1',
    '--num_env_steps',     '2000',
    '--ppo_epoch',         '1',
    '--save_interval',     '9999',
    '--log_interval',      '1',
], 'Smoke test')
if not ok:
    raise RuntimeError('Smoke test failed — fix errors above before full training.')

print(f'\nStarting full training ({GRF_STEPS:,} steps, {N_THREADS} threads)...')
print('-' * 60)
ok = run_train([
    '--experiment_name',   'demo',
    '--n_rollout_threads', str(N_THREADS),
    '--num_env_steps',     str(GRF_STEPS),
    '--save_interval',     '25',
    '--log_interval',      '25',
], 'Full training')
print('-' * 60)
print('Training complete.' if ok else 'Training FAILED — check output above.')

In [ ]:
import shutil, os

RESULTS_BASE = f'{REPO_PATH}/onpolicy/scripts/results'
src = f'{RESULTS_BASE}/Football/academy_3_vs_1_with_keeper/rmappo/demo'
dst = f'{DRIVE_RESULTS}/GRF_3v1'

if os.path.exists(src):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Results saved to: {dst}')
    print('Load in mappo_demo.ipynb: set DEMO_MODE=True and run the GRF cell.')
else:
    print(f'No results found at {src}')
    print('Check that training completed successfully.')

## Results — Training Curve

In [ ]:
import os, glob
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

RESULTS_BASE = f'{REPO_PATH}/onpolicy/scripts/results'

def load_scalar(log_dir, tag):
    nested = os.path.join(log_dir, tag, tag)
    target = nested if os.path.isdir(nested) else log_dir
    ea = EventAccumulator(target)
    ea.Reload()
    if tag in ea.Tags().get('scalars', []):
        ev = ea.Scalars(tag)
        return [e.step for e in ev], [e.value for e in ev]
    return [], []

log_dirs = sorted(glob.glob(
    f'{RESULTS_BASE}/Football/academy_3_vs_1_with_keeper/rmappo/demo/run*/logs'
))

if not log_dirs:
    print('No logs found. Run the training cell first.')
else:
    steps, values = load_scalar(log_dirs[-1], 'average_episode_rewards')
    if steps:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(steps, values, color='#4CAF50', linewidth=2)
        ax.set_xlabel('Environment Steps', fontsize=12)
        ax.set_ylabel('Average Episode Reward', fontsize=12)
        ax.set_title('MAPPO — GRF academy_3_vs_1_with_keeper', fontsize=14, fontweight='bold')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        fig_path = f'{DRIVE_RESULTS}/tc2_grf_curve.png'
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f'Final reward: {values[-1]:.4f}')
        print(f'Figure saved: {fig_path}')
        plt.show()
    else:
        print('No scalar data found in', log_dirs[-1])
        ea_root = EventAccumulator(log_dirs[-1])
        ea_root.Reload()
        print('Available tags:', ea_root.Tags().get('scalars', []))